# Plant Disease Prediction - Model Evaluation on Colab

This notebook demonstrates:
1. Loading the pre-trained model.h5
2. Making predictions on single images
3. Evaluating model performance on test dataset
4. Displaying results with confidence scores

## 1. Setup - Install Dependencies

In [ ]:
# Install required packages
!pip install tensorflow keras pillow numpy matplotlib -q

## 2. Import Libraries

In [ ]:
import os
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")

## 3. Define Class Names (38 Plant Diseases)

In [ ]:
class_names = [
    "Apple___Apple_scab",
    "Apple___Black_rot",
    "Apple___Cedar_apple_rust",
    "Apple___healthy",
    "Blueberry___healthy",
    "Cherry_(including_sour)___Powdery_mildew",
    "Cherry_(including_sour)___healthy",
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "Corn_(maize)___Common_rust_",
    "Corn_(maize)___Northern_Leaf_Blight",
    "Corn_(maize)___healthy",
    "Grape___Black_rot",
    "Grape___Esca_(Black_Measles)",
    "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)",
    "Grape___healthy",
    "Orange___Haunglongbing_(Citrus_greening)",
    "Peach___Bacterial_spot",
    "Peach___healthy",
    "Pepper,_bell___Bacterial_spot",
    "Pepper,_bell___healthy",
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Potato___healthy",
    "Raspberry___healthy",
    "Soybean___healthy",
    "Squash___Powdery_mildew",
    "Strawberry___Leaf_scorch",
    "Strawberry___healthy",
    "Tomato___Bacterial_spot",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot",
    "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato___Target_Spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato___Tomato_mosaic_virus",
    "Tomato___healthy",
]

print(f"Total classes: {len(class_names)}")
print("Classes:", class_names[:5], "...")

## 4. Upload Model (Option A: Upload file) or Load from URL (Option B)

Choose one option below:

In [ ]:
# OPTION A: Upload model.h5 file from your computer
from google.colab import files

print("Upload model.h5 file:")
uploaded = files.upload()

# Get the uploaded filename
model_path = list(uploaded.keys())[0]
print(f"\nUploaded: {model_path}")

## 5. Load the Model

In [ ]:
# Load model
model = keras.models.load_model(model_path, compile=False)
print("Model loaded successfully!")
print(f"Model input shape: {model.input_shape}")
print(f"Model output shape: {model.output_shape}")

## 6. Image Preprocessing Function

In [ ]:
def preprocess_image(img_path):
    """Load and preprocess single image"""
    img = Image.open(img_path).convert('RGB')
    img = img.resize((224, 224))
    img = np.array(img) / 255.0
    return img

print("Preprocessing function ready!")

## 7. Make Prediction on Single Image

In [ ]:
# OPTION B: Upload test image
print("Upload a test image:")
test_image_uploaded = files.upload()
test_image_path = list(test_image_uploaded.keys())[0]

# Make prediction
img = preprocess_image(test_image_path)
img_batch = np.expand_dims(img, axis=0)

pred = model.predict(img_batch, verbose=0)
pred_idx = np.argmax(pred)
confidence = pred[0][pred_idx]
predicted_class = class_names[pred_idx]

print(f"\n{'='*50}")
print(f"Predicted Class: {predicted_class}")
print(f"Confidence: {confidence:.4f} ({confidence*100:.2f}%)")
print(f"{'='*50}")

# Display top 5 predictions
top_5_idx = np.argsort(pred[0])[-5:][::-1]
print("\nTop 5 Predictions:")
for i, idx in enumerate(top_5_idx, 1):
    print(f"{i}. {class_names[idx]}: {pred[0][idx]:.4f} ({pred[0][idx]*100:.2f}%)")

## 8. Visualize Prediction with Image

In [ ]:
# Display image with prediction
plt.figure(figsize=(10, 6))

# Show image
plt.subplot(1, 2, 1)
test_img_display = Image.open(test_image_path)
plt.imshow(test_img_display)
plt.title(f"Input Image")
plt.axis('off')

# Show predictions
plt.subplot(1, 2, 2)
top_5_classes = [class_names[idx] for idx in top_5_idx]
top_5_scores = [pred[0][idx] for idx in top_5_idx]
plt.barh(range(len(top_5_classes)), top_5_scores, color='steelblue')
plt.yticks(range(len(top_5_classes)), top_5_classes, fontsize=9)
plt.xlabel('Confidence Score')
plt.title('Top 5 Predictions')
plt.xlim(0, 1)

# Highlight prediction
for i, v in enumerate(top_5_scores):
    if i == 0:
        plt.text(v + 0.02, i, f'{v:.4f}', color='red', fontweight='bold')
    else:
        plt.text(v + 0.02, i, f'{v:.4f}', fontsize=8)

plt.tight_layout()
plt.show()

## 9. Batch Evaluation on Folder (Optional)

If you have test data organized in class subfolders

In [ ]:
def load_images_from_folder(folder_path):
    """Load all images from folder organized by class subfolders"""
    images = []
    labels = []
    label_to_idx = {name: idx for idx, name in enumerate(class_names)}
    
    folder_path = Path(folder_path)
    
    for class_folder in sorted(folder_path.iterdir()):
        if not class_folder.is_dir():
            continue
        
        class_name = class_folder.name
        if class_name not in label_to_idx:
            print(f"Warning: '{class_name}' not in class_names, skipping.")
            continue
        
        class_idx = label_to_idx[class_name]
        image_count = 0
        
        for img_file in class_folder.glob("*"):
            if img_file.suffix.lower() in ['.jpg', '.jpeg', '.png', '.gif', '.bmp']:
                try:
                    img = preprocess_image(str(img_file))
                    images.append(img)
                    labels.append(class_idx)
                    image_count += 1
                except Exception as e:
                    print(f"Error loading {img_file}: {e}")
        
        if image_count > 0:
            print(f"Loaded {image_count} images from '{class_name}'")
    
    if len(images) == 0:
        print("No images found!")
        return None, None
    
    images = np.array(images)
    labels = np.array(labels)
    return images, labels

print("Batch evaluation function ready!")

In [ ]:
# Upload test data (zip file with folder structure)
# Expected structure:
# test_data.zip
# ├── Apple___Apple_scab/
# │   ├── image1.jpg
# │   ├── image2.jpg
# ├── Apple___healthy/
# │   ├── image3.jpg
# └── ...

print("(Optional) Upload test_data.zip with folder structure:")
print("Structure should be:")
print("  test_data/")
print("    ├── Apple___Apple_scab/")
print("    │   ├── img1.jpg")
print("    ├── Apple___healthy/")
print("    │   ├── img2.jpg")
print("    └── ...")

data_upload = files.upload()
if data_upload:
    zip_file = list(data_upload.keys())[0]
    print(f"\nUploaded: {zip_file}")
    
    import zipfile
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall('.')
    print("Extracted!")

In [ ]:
# Evaluate on folder
test_folder = 'test_data'  # Update this path if different

if os.path.exists(test_folder):
    print(f"Loading images from: {test_folder}")
    print("-" * 50)
    
    images, labels = load_images_from_folder(test_folder)
    
    if images is not None:
        print(f"\nTotal images loaded: {len(images)}")
        print("-" * 50)
        
        # Evaluate
        scores = model.evaluate(images, labels, verbose=1)
        
        print("\n" + "="*50)
        print(f"Loss: {scores[0]:.4f}")
        print(f"Accuracy: {scores[1]:.4f}")
        print("="*50)
else:
    print(f"Folder '{test_folder}' not found. Upload test data first!")

## 10. Make Batch Predictions and Show Results

In [ ]:
if images is not None:
    # Make predictions on all test images
    predictions = model.predict(images, verbose=0)
    pred_classes = np.argmax(predictions, axis=1)
    pred_confidence = np.max(predictions, axis=1)
    
    # Calculate accuracy
    correct = np.sum(pred_classes == labels)
    accuracy = correct / len(labels)
    
    print(f"\nBatch Prediction Results:")
    print(f"Correct: {correct}/{len(labels)}")
    print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"Average Confidence: {np.mean(pred_confidence):.4f}")
    
    # Show first 10 predictions
    print("\nFirst 10 Predictions:")
    print("-" * 80)
    for i in range(min(10, len(labels))):
        true_label = class_names[labels[i]]
        pred_label = class_names[pred_classes[i]]
        match = "✓" if pred_classes[i] == labels[i] else "✗"
        print(f"{match} True: {true_label:40} | Pred: {pred_label:40} | Conf: {pred_confidence[i]:.4f}")

## 11. Confusion Matrix (Optional)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

if images is not None and len(np.unique(labels)) <= 10:  # Only for small number of classes
    # Compute confusion matrix
    cm = confusion_matrix(labels, pred_classes)
    
    # Plot
    plt.figure(figsize=(12, 10))
    plt.imshow(cm, cmap='Blues', aspect='auto')
    plt.colorbar(label='Count')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    
    # Get unique classes in test set
    unique_classes = np.unique(labels)
    class_labels = [class_names[i] for i in unique_classes]
    plt.xticks(range(len(unique_classes)), class_labels, rotation=45, ha='right', fontsize=8)
    plt.yticks(range(len(unique_classes)), class_labels, fontsize=8)
    plt.tight_layout()
    plt.show()
    
    # Classification report
    print("\nClassification Report:")
    print(classification_report(labels, pred_classes, target_names=class_labels, digits=4))
else:
    print("Skipping confusion matrix (too many classes or no data)")

## Summary

This notebook demonstrates:

✓ Loading a pre-trained plant disease model

✓ Making predictions on single images with confidence scores

✓ Batch evaluation on multiple test images

✓ Computing accuracy and other metrics

✓ Visualizing predictions and confusion matrices

You can save this notebook and run it on Google Colab anytime!